In [1]:
%pip install kagglehub
!pip install opendatasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub
import os
import pandas as pd
import numpy as np
import sqlite3
import opendatasets as od

# Importing Dataset

In [3]:
# Download latest version
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset") +'/'

print("Path to dataset files:", path)

df = pd.read_csv(path + "/dataset.csv")

Path to dataset files: C:\Users\avajt\.cache\kagglehub\datasets\maharshipandya\-spotify-tracks-dataset\versions\1/


In [4]:
#Alternate way of importing data, ignore if you did other one
#od.download("https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset/data")

In [5]:
#Alternate cont., ignore
#df = pd.read_csv('dataset.csv')

In [6]:
# Get information on what features are in our dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [7]:
# Drop unnecessary column
df = df.drop(columns=["Unnamed: 0"])

# Creating SQL Database

In [8]:
# Connect to SQLite database
conn = sqlite3.connect("spotify_dataset.db")
cursor = conn.cursor()

In [9]:
# Save dataframe to SQLite
df.to_sql("spotify_tracks", conn, if_exists = "replace", index = False)

114000

In [10]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# Fetch and print table names
tables = cursor.fetchall()
print("Tables in database:", tables)

Tables in database: [('user_playlists',), ('user_tracks',), ('spotify_tracks',)]


In [11]:
with sqlite3.connect("spotify_dataset.db") as conn:
    df_tracks = pd.read_sql_query("SELECT * FROM spotify_tracks;", conn)
df_tracks.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Exploratory Data Analysis

In [12]:
def get_numerical_df(df):
  """
  Returns a dataframe with only the numerical data type columns in a Pandas DataFrame.
  Parameters:
    df: The Pandas DataFrame.
  Returns:
    A Pandas datafram with the numerical columns from df.
  """
  numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
  num_df = df[numerical_cols]
  return num_df

df_num_cols = get_numerical_df(df_tracks)
df_num_cols.head()

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,73,230666,0,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4
1,55,149610,0,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4
2,57,210826,0,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4
3,71,201933,0,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3
4,82,198853,0,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4


In [13]:
corr_matrix = df_num_cols.corr()

In [14]:
# Create correlatoin matrix heatmap
from plotly import express as px
fig = px.imshow(corr_matrix,
                x=corr_matrix.columns,
                y=corr_matrix.index,
                title="Correlation Matrix Heatmap")

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    width=800,
    height=800
)

fig.show()

From the correlation matrix, we notice a higher correlation between:
* energy & loudness
* danceability & valence (how positive/negative it sounds)
* explicit (considered innapropriate) & speechiness (how much talking there is)
These correlations may be useful when selecting features for machine learning algorithms in next steps (i.e. how songs are similar, why users like specific songs).

In [15]:
popular = df[df["popularity"] >= 70]
popular.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,False,0.407,0.1470,2,-8.822,1,0.0355,0.8570,0.000003,0.0913,0.0765,141.284,3,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,0.4440,11,-9.331,1,0.0417,0.5590,0.000000,0.0973,0.7120,150.960,4,acoustic


In [16]:
fig = px.scatter(popular, x="energy", y="popularity", title="Popularity vs Energy for Popular Songs")

fig.update_layout(
    width=800,
    height=600)

fig.show()

From this scatterplot, for the songs that have popularity of at least 70, there does not appear to be a strong correlation between the energy of a song and how popular it is. This could mean that the most popular songs that people listen to on Spotify are not necessarily the radio hits that come to mind when you hear "popular", and people want recommendations for songs of all kinds of energy levels.

### genre_plot
Bea made a function to plot a quantitative variable by genre using a box plot. This is helpful because genre is the main qualitative variable we'll use, so it's useful to understand how it correlates with the quantitative variables.

In [17]:
genres = df_tracks['track_genre'].unique()

In [18]:
# Import Exploratory Data Analysis's genre_plot method to plot the popularity vs genre
from eda import genre_plot

fig = genre_plot('spotify_dataset.db', genres[:25], 'popularity')
fig.update_layout(width = 800, height = 500)
fig

In [ ]:
from music_rec import remove_duplicates
filtered_spotify = remove_duplicates(df_tracks, ['track_name', 'artists'])

fig = px.histogram(filtered_spotify, 'track_genre', color='track_genre')
fig

There are 32656 duplicate rows based on track_name and artists. Dropping them now...
After dropping duplicates, there are 81344 rows left.


Odd genres:
- bluegrass
- breakbeat
- cantopop
- chicago-house
- deep-house
- detroit-techno
- happy
- kids vs. children
- singer-songwriter

j-rock has 449 while rock has 167 songs. 
The genre classification of the songs seems to be a little off, but to try to reclassify all the songs genres would've been nearly impossible.

We see lots of data in these random subcategories, but not a lot of data in more modern popular categories like indie, pop, rock, and punk. This means for users who listen to more current music, our model will struggle to suggest as many songs. 

In [ ]:
def get_user_df():
    """
    no args
    returns df with user songs that are also in kaggle dataset
    """
    
    user_df = pd.read_sql_query('''
        SELECT * 
        FROM user_tracks INNER JOIN spotify_tracks 
        ON user_tracks.track_id = spotify_tracks.track_id;''', conn)
    print(user_df.info())
    user_df = user_df.drop_duplicates(subset='track_name', keep='first')
    print(user_df.info())

In [ ]:
get_user_df()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5530 entries, 0 to 5529
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   playlist_id        5530 non-null   object 
 1   track_id           5530 non-null   object 
 2   track_explicit     5530 non-null   int64  
 3   track_album_id     5530 non-null   object 
 4   track_artists      5530 non-null   object 
 5   track_duration_ms  5530 non-null   int64  
 6   track_href         5530 non-null   object 
 7   track_name         5530 non-null   object 
 8   track_popularity   5530 non-null   int64  
 9   username           5530 non-null   object 
 10  track_id           5530 non-null   object 
 11  artists            5530 non-null   object 
 12  album_name         5530 non-null   object 
 13  track_name         5530 non-null   object 
 14  popularity         5530 non-null   int64  
 15  duration_ms        5530 non-null   int64  
 16  explicit           5530 

In [ ]:
conn.close()